In [56]:
%pip install pydantic-ai

Note: you may need to restart the kernel to use updated packages.


In [61]:
import os
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.models.google import GoogleModel
from pydantic import BaseModel, Field
from pydantic_ai.providers.google import GoogleProvider
from pydantic_ai.capabilities import WebSearch

GOOGLE_API_KEY = os.getenv("gemini")

class ImageAnalysisResult(BaseModel):
    summary: str = Field(description="이미지 전체 요약")
    objects: list[str] = Field(description="이미지에서 감지된 주요 객체 목록")
    category: str = Field(description="이미지의 분류 카테고리")


In [62]:
# 2. LangChain과 동일하게 503/429 발생 시 최대 6회 자동 재시도하도록 설정
provider = GoogleProvider(
    api_key=GOOGLE_API_KEY
)

model = GoogleModel(model_name='gemini-3.1-flash-lite', provider=provider)

# 에이전트 생성
agent = Agent(
    model,
    instructions="햇갈리면 웹검색을 쓸것 꼭 쓸 필요는 없음",
    capabilities=[WebSearch(native=False, local='duckduckgo')],
    output_type=ImageAnalysisResult
)

with open("sample_image.jpg", "rb") as f:
    image_bytes = f.read()
result = await agent.run(
[
    "이 이미지를 분석해라",
    BinaryContent(data=image_bytes, media_type="image/jpeg")
]
)

result.output

ImageAnalysisResult(summary="MAC 브랜드의 핑크색 립스틱 제품 사진입니다. 검은색 배경에 립스틱이 수직으로 서 있는 형태이며, 제품 본체에는 'MAC' 로고가 새겨져 있습니다.", objects=['립스틱 (MAC 브랜드)'], category='화장품/뷰티')

In [63]:
output_data = result.output
print(output_data.summary)
print(output_data.objects)
print(output_data.category)

MAC 브랜드의 핑크색 립스틱 제품 사진입니다. 검은색 배경에 립스틱이 수직으로 서 있는 형태이며, 제품 본체에는 'MAC' 로고가 새겨져 있습니다.
['립스틱 (MAC 브랜드)']
화장품/뷰티
